# 통합인사정보 데이터 전처리

### 라이브러리 선언

In [1]:
import pandas as pd
from datetime import date
from pathlib import Path
import os
from dotenv import load_dotenv

print(f'pandas 버전: {pd.__version__}')
print('라이브러리 로딩 완료!')

pandas 버전: 3.0.2
라이브러리 로딩 완료!


### 환경 설정 *** 경로 및 설정값 확인 필요

In [2]:
load_dotenv()

INPUT_DIR  = Path(os.getenv('INPUT_DIR',  'dataset'))
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('경로 설정:')
print(f'  입력 디렉토리: {INPUT_DIR}')
print(f'  출력 디렉토리: {OUTPUT_DIR}')

경로 설정:
  입력 디렉토리: dataset
  출력 디렉토리: output


In [3]:
TODAY = date.today()

# 이상치 허용 범위
MIN_AGE,          MAX_AGE          = 18, 80
MIN_SALARY,       MAX_SALARY       = 20_000_000, 500_000_000
MIN_OVERTIME,     MAX_OVERTIME     = 0, 52
MIN_UNUSED_LEAVE, MAX_UNUSED_LEAVE = 0, 30
MIN_GPA,          MAX_GPA          = 0.0, 4.5
MIN_SCORE,        MAX_SCORE        = 0, 100
MIN_TOEIC,        MAX_TOEIC        = 0, 990

# 조직 고정값
DEPARTMENTS = ['개발부', '인사부', '영업부', '마케팅부', '기획부']

DEPT_TEAM_MAP = {
    '개발부':   ['백엔드팀', '프론트팀', 'AI팀', '인프라팀'],
    '인사부':   ['채용팀', '교육팀'],
    '영업부':   ['국내영업팀', '해외영업팀'],
    '마케팅부': ['디지털마케팅팀', '브랜드팀'],
    '기획부':   ['전략기획팀', '사업기획팀'],
}

DEPT_LEVEL_MAP = {
    '개발부':   1,
    '인사부':   3,
    '영업부':   1,
    '마케팅부': 1,
    '기획부':   1,
}

GRADE_LEVEL_MAP = {
    '사원': 1, '대리': 1, '과장': 1,
    '차장': 2, '부장': 2,
    '이사': 3, '사장': 3,
}

POSITIONS        = ['팀원', '팀장', '본부장', '대표이사']
PERF_GRADES      = ['S', 'A', 'B', 'C', 'D', 'F']
INSURANCE_VALUES = ['가입', '미가입']
SUBSIDY_VALUES   = ['해당', '비해당']
PERF_YEARS       = [2020, 2021, 2022, 2023, 2024]

print('설정된 허용 범위:')
print(f'  나이        : {MIN_AGE} ~ {MAX_AGE}세')
print(f'  연봉        : {MIN_SALARY:,} ~ {MAX_SALARY:,}원')
print(f'  잔업시간    : {MIN_OVERTIME} ~ {MAX_OVERTIME}시간')
print(f'  미사용휴가  : {MIN_UNUSED_LEAVE} ~ {MAX_UNUSED_LEAVE}일')
print(f'  학점        : {MIN_GPA} ~ {MAX_GPA}')
print(f'  성과점수    : {MIN_SCORE} ~ {MAX_SCORE}')
print(f'  TOEIC       : {MIN_TOEIC} ~ {MAX_TOEIC}')
print(f'\n조직 고정값:')
print(f'  부서        : {DEPARTMENTS}')
print(f'  직책        : {POSITIONS}')
print(f'  인사고과    : {PERF_GRADES}')

설정된 허용 범위:
  나이        : 18 ~ 80세
  연봉        : 20,000,000 ~ 500,000,000원
  잔업시간    : 0 ~ 52시간
  미사용휴가  : 0 ~ 30일
  학점        : 0.0 ~ 4.5
  성과점수    : 0 ~ 100
  TOEIC       : 0 ~ 990

조직 고정값:
  부서        : ['개발부', '인사부', '영업부', '마케팅부', '기획부']
  직책        : ['팀원', '팀장', '본부장', '대표이사']
  인사고과    : ['S', 'A', 'B', 'C', 'D', 'F']


# 1. 데이터 로딩

In [4]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

if not csv_files:
    print(f'CSV 파일 없음: {INPUT_DIR}')
    raise SystemExit

# 파일 이름을 키로, DataFrame을 값으로 저장 (파일끼리 합치지 않음)
dfs = {}
for path in csv_files:
    # dtype=object: 모든 컬럼을 문자/숫자 혼합 저장 가능하게 읽음
    df = pd.read_csv(path, encoding='utf-8-sig', dtype=object)
    dfs[path.stem] = df
    print(f'  로딩: {path.name}  ({len(df):,}행 / {len(df.columns)}열)')

print(f'\n로딩 완료! 총 {len(dfs)}개 파일')

  로딩: 급여정보.csv  (2,000행 / 7열)
  로딩: 기본인사정보.csv  (2,000행 / 30열)
  로딩: 역량성과.csv  (2,000행 / 13열)
  로딩: 통합인사정보.csv  (2,000행 / 48열)

로딩 완료! 총 4개 파일


### 에러 로그 초기화

In [5]:
# 에러 내용을 저장하는 리스트 (파일마다 초기화됨)
_errors = []
# 삭제할 행 번호를 저장하는 집합 (파일마다 초기화됨)
drop_rows = set()
# 주민번호에서 파싱한 생년월일 — validate_rrn → validate_gender/birth/age/hire에서 사용
valid_rrn = {}
# 유효한 입사일 — validate_hire → validate_tenure/retire/perf에서 사용
valid_hire = {}

def log(row, emp_id, column, original_value, reason):
    _errors.append({
        '행': row,
        '사원번호': emp_id,
        '컬럼': column,
        '원본값': original_value,
        '사유': reason
    })

print('에러 로그 초기화 완료!')

에러 로그 초기화 완료!


### 헬퍼 함수 정의

In [6]:
def calc_age(birth):
    age = TODAY.year - birth.year
    if (TODAY.month, TODAY.day) < (birth.month, birth.day):
        age -= 1
    return age

def tenure_years(hire_date):
    days = (TODAY - hire_date).days
    return round(days / 365.25, 2)

def parse_date(text):
    try:
        text = str(text).strip()
        # 빈값이거나 nan/None 문자열이면 None 반환
        if not text or text in ('nan', 'NaN', 'None', '미입력'):
            return None
        return pd.to_datetime(text).date()
    except Exception:
        return None

def birth_from_rrn(rrn):
    try:
        digits = rrn.replace('-', '')
        yy = int(digits[0:2])
        mm = int(digits[2:4])
        dd = int(digits[4:6])
        seven = int(digits[6])
        if seven in (1, 2):
            year = 1900 + yy
        elif seven in (3, 4):
            year = 2000 + yy
        else:
            return None
        return date(year, mm, dd)
    except Exception:
        return None

def rrn_checksum_ok(rrn):
    digits = rrn.replace('-', '')
    if len(digits) != 13 or not digits.isdigit():
        return False
    weights = [2, 3, 4, 5, 6, 7, 8, 9, 2, 3, 4, 5]
    total = 0
    for i in range(12):
        total += int(digits[i]) * weights[i]
    check = (11 - total % 11) % 10
    return check == int(digits[12])

def parse_array(cell_value):
    if not cell_value:
        return []
    items = []
    for item in str(cell_value).split(','):
        item = item.strip()
        if item:
            items.append(item)
    return items

def to_num(cell_value, column, row, emp_id):
    # int(float(...))으로 '85.0' 형태 문자열도 처리 가능
    try:
        return int(float(str(cell_value).strip()))
    except Exception:
        log(row, emp_id, column, cell_value, '숫자 변환 불가')
        return None

# ── 형식 검사 함수 ──────────────────────────────────────

def is_valid_emp_id(emp_id):
    # 길이가 7자리인지 확인 (EMP + 숫자 4개)
    if len(emp_id) != 7:
        return False
    # 앞 3자리가 EMP인지 확인
    if emp_id[:3] != 'EMP':
        return False
    # 뒤 4자리가 모두 숫자인지 확인
    if not emp_id[3:].isdigit():
        return False
    return True

def is_valid_name(name):
    # 이름은 글자(한글·영문·한자)와 공백만 허용, 숫자·특수문자 불가
    for char in name:
        if not char.isalpha() and char != ' ':
            return False
    return True

def is_valid_rrn_format(rrn):
    # 형식: 숫자 6자리 + '-' + 숫자 7자리 (총 14자리)
    if len(rrn) != 14:
        return False
    if rrn[6] != '-':
        return False
    if not rrn[:6].isdigit():
        return False
    if not rrn[7:].isdigit():
        return False
    return True

def is_valid_email(email):
    # @가 정확히 1개 있어야 함
    if email.count('@') != 1:
        return False
    parts = email.split('@')
    local = parts[0]   # @ 앞부분
    domain = parts[1]  # @ 뒷부분
    # 앞부분이 비어 있으면 안 됨
    if not local:
        return False
    # 뒷부분에 .이 있어야 함
    if '.' not in domain:
        return False
    # .이 도메인 맨 끝에 있으면 안 됨
    if domain.endswith('.'):
        return False
    return True

def is_valid_phone(phone):
    # 0으로 시작해야 함 (02, 010, 011 등)
    if not phone.startswith('0'):
        return False
    # '-'가 정확히 2개 있어야 함
    if phone.count('-') != 2:
        return False
    # 각 구역이 모두 숫자이어야 함
    parts = phone.split('-')
    for part in parts:
        if not part.isdigit():
            return False
    return True

def is_valid_bank(bank):
    # 은행명에 숫자나 특수문자가 있으면 안 됨 (한글·영문·공백만 허용)
    for char in bank:
        if not char.isalpha() and char != ' ':
            return False
    return True

print('헬퍼 함수 정의 완료!')

헬퍼 함수 정의 완료!


# 2. 데이터 정제

### 2-1. 식별자 (사원번호 · 이름)

In [7]:
def validate_empid(df):
    # 이 파일에 사원번호 컬럼이 없으면 건너뜀
    if '사원번호' not in df.columns:
        return

    seen_emp = {}  # 이 파일 안에서의 사원번호 중복 체크 (파일별로 독립)

    for row, raw in df['사원번호'].items():
        emp_id = str(raw).strip() if raw else ''

        if not emp_id or emp_id in ('nan', 'NaN'):
            log(row, emp_id, '사원번호', emp_id, '결측')
            drop_rows.add(row)
        elif not is_valid_emp_id(emp_id):
            log(row, emp_id, '사원번호', emp_id, '형식 오류 (EMP+4자리)')
            drop_rows.add(row)
        elif emp_id in seen_emp:
            log(row, emp_id, '사원번호', emp_id, '중복 → 나중 행 제거')
            drop_rows.add(row)
        else:
            seen_emp[emp_id] = row

        df.at[row, '사원번호'] = emp_id

print('validate_empid 정의 완료')

validate_empid 정의 완료


In [8]:
def validate_name(df):
    # 이 파일에 이름 컬럼이 없으면 건너뜀
    if '이름' not in df.columns:
        return

    for row, raw in df['이름'].items():
        name = str(raw).strip() if raw else ''
        emp_id = df.at[row, '사원번호']

        if not name or name in ('nan', 'NaN'):
            log(row, emp_id, '이름', name, '결측')
            drop_rows.add(row)
        elif not is_valid_name(name):
            log(row, emp_id, '이름', name, '숫자/특수문자 포함')
            drop_rows.add(row)

print('validate_name 정의 완료')

validate_name 정의 완료


### 2-2. 개인정보 (주민번호 · 성별 · 나이 · 생년월일 · 병역)

In [9]:
def validate_rrn(df):
    if '주민등록번호' not in df.columns:
        return

    for row, raw in df['주민등록번호'].items():
        rrn = str(raw).strip() if raw else ''
        emp_id = df.at[row, '사원번호']

        if not rrn or rrn in ('nan', 'NaN'):
            log(row, emp_id, '주민등록번호', rrn, '결측')
            df.at[row, '주민등록번호'] = '미입력'
            continue

        if not is_valid_rrn_format(rrn):
            log(row, emp_id, '주민등록번호', rrn, '형식 오류')
            df.at[row, '주민등록번호'] = '미입력'
            continue

        birth = birth_from_rrn(rrn)
        if birth is None:
            log(row, emp_id, '주민등록번호', rrn, '날짜 파싱 불가')
            df.at[row, '주민등록번호'] = '미입력'
            continue

        if not (MIN_AGE <= calc_age(birth) <= MAX_AGE):
            log(row, emp_id, '주민등록번호', rrn, f'생년월일 범위 초과 (나이={calc_age(birth)})')
            df.at[row, '주민등록번호'] = '미입력'
            continue

        if not rrn_checksum_ok(rrn):
            log(row, emp_id, '주민등록번호', rrn, '체크섬 오류')
            df.at[row, '주민등록번호'] = '미입력'
            continue

        # valid_rrn: 전역 딕셔너리 — 이후 validate_gender/birth/age/hire에서 사용
        valid_rrn[row] = birth

print('validate_rrn 정의 완료')

validate_rrn 정의 완료


In [10]:
def validate_gender(df):
    if '성별' not in df.columns:
        return

    for row in df.index:
        rrn_str = ''
        if '주민등록번호' in df.columns:
            rrn_str = str(df.at[row, '주민등록번호']).replace('-', '')
        current_gender = str(df.at[row, '성별']).strip() if df.at[row, '성별'] else ''

        if rrn_str and rrn_str not in ('미입력', 'nan', 'NaN') and len(rrn_str) >= 7 and rrn_str.isdigit():
            correct_gender = '남' if int(rrn_str[6]) in (1, 3) else '여'
            if current_gender != correct_gender:
                log(row, df.at[row, '사원번호'], '성별', current_gender, f'주민번호와 불일치 → {correct_gender} 교정')
                df.at[row, '성별'] = correct_gender
        elif current_gender not in ('남', '여'):
            log(row, df.at[row, '사원번호'], '성별', current_gender, '결측/이상치 + 주민번호 없음')
            df.at[row, '성별'] = '미입력'

print('validate_gender 정의 완료')

validate_gender 정의 완료


In [11]:
def validate_birth(df):
    if '생년월일' not in df.columns:
        return

    for row in df.index:
        rrn_birth = valid_rrn.get(row)
        birth_str = str(df.at[row, '생년월일']).strip() if df.at[row, '생년월일'] else ''
        if birth_str in ('nan', 'NaN'):
            birth_str = ''

        birth = parse_date(birth_str)

        if rrn_birth:
            if birth is None or not (MIN_AGE <= calc_age(birth) <= MAX_AGE):
                if birth is not None:
                    log(row, df.at[row, '사원번호'], '생년월일', birth_str, '범위 초과 → 주민번호로 교정')
                df.at[row, '생년월일'] = rrn_birth.strftime('%Y-%m-%d')
            else:
                df.at[row, '생년월일'] = birth.strftime('%Y-%m-%d')
        else:
            if birth is None:
                log(row, df.at[row, '사원번호'], '생년월일', birth_str, '결측/파싱불가 + 주민번호 없음')
                df.at[row, '생년월일'] = '미입력'
            elif not (MIN_AGE <= calc_age(birth) <= MAX_AGE):
                log(row, df.at[row, '사원번호'], '생년월일', birth_str, '범위 초과')
                df.at[row, '생년월일'] = '미입력'
            else:
                df.at[row, '생년월일'] = birth.strftime('%Y-%m-%d')

print('validate_birth 정의 완료')

validate_birth 정의 완료


In [12]:
def validate_age(df):
    if '나이' not in df.columns:
        return

    for row in df.index:
        birth_str = str(df.at[row, '생년월일']) if '생년월일' in df.columns else '미입력'
        if birth_str in ('nan', 'NaN'):
            birth_str = '미입력'

        birth = parse_date(birth_str) if birth_str != '미입력' else None

        if birth:
            df.at[row, '나이'] = calc_age(birth)
        elif valid_rrn.get(row):
            df.at[row, '나이'] = calc_age(valid_rrn[row])
        else:
            log(row, df.at[row, '사원번호'], '나이', df.at[row, '나이'], '생년월일·주민번호 없음')
            df.at[row, '나이'] = '미입력'

print('validate_age 정의 완료')

validate_age 정의 완료


In [13]:
def validate_military(df):
    if '병역' not in df.columns:
        return

    for row in df.index:
        gender = str(df.at[row, '성별']).strip() if '성별' in df.columns else ''
        military = str(df.at[row, '병역']).strip() if df.at[row, '병역'] else ''
        if military in ('nan', 'NaN'):
            military = ''

        if gender == '여':
            df.at[row, '병역'] = '해당없음'
        elif not military:
            log(row, df.at[row, '사원번호'], '병역', military, '결측')
            df.at[row, '병역'] = '미입력'

print('validate_military 정의 완료')

validate_military 정의 완료


### 2-3. 근무 정보 (입사일 · 근속기간)

In [14]:
def validate_hire(df):
    if '입사일' not in df.columns:
        return

    for row, raw in df['입사일'].items():
        hire_str = str(raw).strip() if raw else ''
        if hire_str in ('nan', 'NaN'):
            hire_str = ''
        hire_date = parse_date(hire_str)
        emp_id = df.at[row, '사원번호']

        birth_str = str(df.at[row, '생년월일']) if '생년월일' in df.columns else '미입력'
        if birth_str in ('nan', 'NaN'):
            birth_str = '미입력'
        birth = parse_date(birth_str) if birth_str != '미입력' else None

        earliest_hire = date(birth.year + MIN_AGE, birth.month, birth.day) if birth else None

        if hire_date is None:
            log(row, emp_id, '입사일', hire_str, '결측/파싱불가')
            df.at[row, '입사일'] = '미입력'
        elif hire_date > TODAY:
            log(row, emp_id, '입사일', hire_str, '현재 날짜 초과')
            df.at[row, '입사일'] = '미입력'
        elif earliest_hire and hire_date < earliest_hire:
            log(row, emp_id, '입사일', hire_str, '만 18세 이전')
            df.at[row, '입사일'] = '미입력'
        else:
            # valid_hire: 전역 딕셔너리 — 이후 validate_tenure/retire/perf에서 사용
            valid_hire[row] = hire_date
            df.at[row, '입사일'] = hire_date.strftime('%Y-%m-%d')

print('validate_hire 정의 완료')

validate_hire 정의 완료


In [15]:
def validate_tenure(df):
    if '근속기간' not in df.columns:
        return

    for row in df.index:
        hire_date = valid_hire.get(row)
        if hire_date:
            df.at[row, '근속기간'] = tenure_years(hire_date)
        else:
            log(row, df.at[row, '사원번호'], '근속기간', df.at[row, '근속기간'], '입사일 없어 계산 불가')
            df.at[row, '근속기간'] = '미입력'

print('validate_tenure 정의 완료')

validate_tenure 정의 완료


### 2-4. 학력 · 경력

In [16]:
def validate_edu(df):
    if '학력' not in df.columns:
        return

    for row in df.index:
        edu = str(df.at[row, '학력']).strip() if df.at[row, '학력'] else ''
        if edu in ('nan', 'NaN'):
            edu = ''
        univ = str(df.at[row, '출신대학']).strip() if '출신대학' in df.columns and df.at[row, '출신대학'] else ''
        if univ in ('nan', 'NaN'):
            univ = ''
        emp_id = df.at[row, '사원번호']

        if edu == '대학원졸':
            log(row, emp_id, '학력', edu, '대학원졸 허용 안 함')
            edu = '미입력'
            df.at[row, '학력'] = '미입력'

        if not edu or edu == '미입력':
            if univ and univ != '미입력':
                edu = '전문대졸' if '전문' in univ else '대졸'
                df.at[row, '학력'] = edu
            else:
                log(row, emp_id, '학력', edu, '결측 + 출신대학 없음')
                df.at[row, '학력'] = '미입력'
                edu = '미입력'

        if edu == '고졸':
            if univ:
                log(row, emp_id, '출신대학', univ, '고졸인데 출신대학 존재 → 삭제')
                df.at[row, '출신대학'] = ''
        elif edu in ('대졸', '전문대졸') and '출신대학' in df.columns:
            if not univ:
                log(row, emp_id, '출신대학', univ, '결측')
                df.at[row, '출신대학'] = '미입력'
            else:
                is_junior = '전문' in univ
                if edu == '전문대졸' and not is_junior:
                    log(row, emp_id, '출신대학', univ, '학력=전문대졸인데 일반대학교명')
                    df.at[row, '출신대학'] = '미입력'
                elif edu == '대졸' and is_junior:
                    log(row, emp_id, '출신대학', univ, '학력=대졸인데 전문대학교명')
                    df.at[row, '출신대학'] = '미입력'

print('validate_edu 정의 완료')

validate_edu 정의 완료

In [17]:
def validate_career(df):
    # 학점
    if '학점' in df.columns:
        for row in df.index:
            edu = str(df.at[row, '학력']).strip() if '학력' in df.columns else ''
            gpa_raw = df.at[row, '학점']

            if edu == '고졸':
                df.at[row, '학점'] = ''
                continue

            try:
                gpa = float(str(gpa_raw).strip())
            except Exception:
                gpa = None
                log(row, df.at[row, '사원번호'], '학점', gpa_raw, '숫자 변환 불가')

            if gpa is None:
                df.at[row, '학점'] = '미입력'
            elif not (MIN_GPA <= gpa <= MAX_GPA):
                log(row, df.at[row, '사원번호'], '학점', gpa_raw, f'범위 초과 ({MIN_GPA}~{MAX_GPA})')
                df.at[row, '학점'] = '미입력'

    # 채용경로 / 계약형태
    for col in ['채용경로', '계약형태']:
        if col not in df.columns:
            continue
        for row, raw in df[col].items():
            cell_value = str(raw).strip() if raw else ''
            if cell_value in ('nan', 'NaN'):
                cell_value = ''
            if not cell_value:
                log(row, df.at[row, '사원번호'], col, cell_value, '결측')
                df.at[row, col] = '미입력'

    # 이전직장명 / 이전최종직급 / 이전담당업무
    if '채용경로' in df.columns:
        for row in df.index:
            hire_route = str(df.at[row, '채용경로']).strip()
            for col in ['이전직장명', '이전최종직급', '이전담당업무']:
                if col not in df.columns:
                    continue
                cell_value = str(df.at[row, col]).strip() if df.at[row, col] else ''
                if cell_value in ('nan', 'NaN'):
                    cell_value = ''
                if hire_route == '경력' and not cell_value:
                    log(row, df.at[row, '사원번호'], col, cell_value, '채용경로=경력인데 결측')
                    df.at[row, col] = '미입력'

print('validate_career 정의 완료')

validate_career 정의 완료


### 2-5. 조직 정보 (부서 · 팀 · 직급 · 직책 · 레벨)

In [18]:
def validate_dept(df):
    # 회사명 / 사업장위치
    for col in ['회사명', '사업장위치']:
        if col not in df.columns:
            continue
        for row, raw in df[col].items():
            cell_value = str(raw).strip() if raw else ''
            if cell_value in ('nan', 'NaN'):
                cell_value = ''
            if not cell_value:
                log(row, df.at[row, '사원번호'], col, cell_value, '결측')
                df.at[row, col] = '미입력'

    # 부서 — 이 파일에 없으면 이하 검증 전부 건너뜀
    if '부서' not in df.columns:
        return

    for row, raw in df['부서'].items():
        dept = str(raw).strip() if raw else ''
        if dept in ('nan', 'NaN'):
            dept = ''
        if not dept or dept not in DEPARTMENTS:
            log(row, df.at[row, '사원번호'], '부서', dept, '결측/이상치 → 행 제거')
            drop_rows.add(row)

    # 팀
    if '팀' in df.columns:
        for row in df.index:
            dept = str(df.at[row, '부서']).strip()
            team = str(df.at[row, '팀']).strip() if df.at[row, '팀'] else ''
            if team in ('nan', 'NaN'):
                team = ''
            if not team:
                log(row, df.at[row, '사원번호'], '팀', team, '결측')
                df.at[row, '팀'] = '미입력'
            elif dept in DEPT_TEAM_MAP and team not in DEPT_TEAM_MAP[dept]:
                log(row, df.at[row, '사원번호'], '팀', team, f'부서({dept})-팀 매핑 불일치')
                df.at[row, '팀'] = '미입력'

    # 부서레벨
    if '부서레벨' in df.columns:
        for row in df.index:
            dept = str(df.at[row, '부서']).strip()
            if dept in DEPT_LEVEL_MAP:
                df.at[row, '부서레벨'] = DEPT_LEVEL_MAP[dept]
            else:
                log(row, df.at[row, '사원번호'], '부서레벨', df.at[row, '부서레벨'], '부서 없음 → 행 제거')
                drop_rows.add(row)

print('validate_dept 정의 완료')

validate_dept 정의 완료


In [19]:
def validate_grade(df):
    if '직급' not in df.columns:
        return

    for row, raw in df['직급'].items():
        grade = str(raw).strip() if raw else ''
        if grade in ('nan', 'NaN'):
            grade = ''
        if not grade or grade not in GRADE_LEVEL_MAP:
            log(row, df.at[row, '사원번호'], '직급', grade, '결측/이상치 → 행 제거')
            drop_rows.add(row)

    if '직책' in df.columns:
        ceo_row = None  # 대표이사 중복 체크 (함수 내 지역 변수)
        for row in df.index:
            grade = str(df.at[row, '직급']).strip()
            position = str(df.at[row, '직책']).strip() if df.at[row, '직책'] else ''
            if position in ('nan', 'NaN'):
                position = ''

            if not position or position not in POSITIONS:
                log(row, df.at[row, '사원번호'], '직책', position, '결측/이상치')
                df.at[row, '직책'] = '미입력'
                continue

            if grade == '사장' and position != '대표이사':
                log(row, df.at[row, '사원번호'], '직책', position, '직급=사장인데 직책≠대표이사 → 교정')
                df.at[row, '직책'] = '대표이사'
                position = '대표이사'

            if position == '대표이사':
                if ceo_row is None:
                    ceo_row = row
                else:
                    log(row, df.at[row, '사원번호'], '직책', position, '대표이사 중복 (1명만 허용)')
                    df.at[row, '직책'] = '미입력'

    if '직급레벨' in df.columns:
        for row in df.index:
            grade = str(df.at[row, '직급']).strip()
            if grade in GRADE_LEVEL_MAP:
                df.at[row, '직급레벨'] = GRADE_LEVEL_MAP[grade]
            else:
                log(row, df.at[row, '사원번호'], '직급레벨', df.at[row, '직급레벨'], '직급 없음 → 행 제거')
                drop_rows.add(row)

print('validate_grade 정의 완료')

validate_grade 정의 완료


### 2-6. 퇴직 · 연락처 · 급여

In [20]:
def validate_retire(df):
    if '퇴직구분' not in df.columns:
        return

    for row in df.index:
        retire_type = str(df.at[row, '퇴직구분']).strip() if df.at[row, '퇴직구분'] else ''
        if retire_type in ('nan', 'NaN'):
            retire_type = ''
        retire_date_str = ''
        if '퇴직일자' in df.columns and df.at[row, '퇴직일자']:
            retire_date_str = str(df.at[row, '퇴직일자']).strip()
            if retire_date_str in ('nan', 'NaN'):
                retire_date_str = ''
        emp_id = df.at[row, '사원번호']

        hire_date  = valid_hire.get(row)
        retire_date = parse_date(retire_date_str)

        if retire_date_str and retire_date is None:
            log(row, emp_id, '퇴직일자', retire_date_str, '날짜 파싱 불가')
            df.at[row, '퇴직일자'] = '미입력'
            retire_date_str = ''

        if retire_date:
            if hire_date and retire_date < hire_date:
                log(row, emp_id, '퇴직일자', retire_date_str, '입사일 이전')
                df.at[row, '퇴직일자'] = '미입력'
                retire_date = None
                retire_date_str = ''
            elif retire_date > TODAY:
                log(row, emp_id, '퇴직일자', retire_date_str, '현재 날짜 초과')
                df.at[row, '퇴직일자'] = '미입력'
                retire_date = None
                retire_date_str = ''

        if retire_date and not retire_type:
            log(row, emp_id, '퇴직구분', retire_type, '퇴직일자 있는데 퇴직구분 없음')
            df.at[row, '퇴직구분'] = '미입력'

        if retire_type and not retire_date_str and '퇴직일자' in df.columns:
            log(row, emp_id, '퇴직일자', retire_date_str, '퇴직구분 있는데 퇴직일자 없음')
            df.at[row, '퇴직일자'] = '미입력'

print('validate_retire 정의 완료')

validate_retire 정의 완료


In [21]:
def validate_contact(df):
    seen_email = {}  # 이 파일 안에서의 이메일 중복 체크

    if '이메일' in df.columns:
        for row, raw in df['이메일'].items():
            email = str(raw).strip() if raw else ''
            if email in ('nan', 'NaN'):
                email = ''
            emp_id = df.at[row, '사원번호']

            if not email:
                log(row, emp_id, '이메일', email, '결측')
                df.at[row, '이메일'] = '미입력'
            elif not is_valid_email(email):
                log(row, emp_id, '이메일', email, '형식 오류')
                df.at[row, '이메일'] = '미입력'
            elif email in seen_email:
                log(row, emp_id, '이메일', email, '중복')
                df.at[row, '이메일'] = '미입력'
            else:
                seen_email[email] = row

    if '전화번호' in df.columns:
        for row, raw in df['전화번호'].items():
            phone = str(raw).strip() if raw else ''
            if phone in ('nan', 'NaN'):
                phone = ''
            if not phone:
                log(row, df.at[row, '사원번호'], '전화번호', phone, '결측')
                df.at[row, '전화번호'] = '미입력'
            elif not is_valid_phone(phone):
                log(row, df.at[row, '사원번호'], '전화번호', phone, '형식 오류')
                df.at[row, '전화번호'] = '미입력'

    if '주소' in df.columns:
        for row, raw in df['주소'].items():
            address = str(raw).strip() if raw else ''
            if address in ('nan', 'NaN'):
                address = ''
            if not address:
                log(row, df.at[row, '사원번호'], '주소', address, '결측')
                df.at[row, '주소'] = '미입력'

print('validate_contact 정의 완료')

validate_contact 정의 완료


In [22]:
def validate_salary(df):
    for col, min_val, max_val in [
        ('연봉',          MIN_SALARY,       MAX_SALARY),
        ('잔업시간',       MIN_OVERTIME,     MAX_OVERTIME),
        ('미사용휴가일수', MIN_UNUSED_LEAVE, MAX_UNUSED_LEAVE),
    ]:
        if col not in df.columns:
            continue
        for row, raw in df[col].items():
            val = to_num(raw, col, row, df.at[row, '사원번호'])
            if val is None:
                df.at[row, col] = '미입력'
            elif not (min_val <= val <= max_val):
                log(row, df.at[row, '사원번호'], col, raw, f'범위 초과 ({min_val:,}~{max_val:,})')
                df.at[row, col] = '미입력'

    if '급여은행' in df.columns:
        for row, raw in df['급여은행'].items():
            bank = str(raw).strip() if raw else ''
            if bank in ('nan', 'NaN'):
                bank = ''
            if not bank:
                log(row, df.at[row, '사원번호'], '급여은행', bank, '결측')
                df.at[row, '급여은행'] = '미입력'
            elif not is_valid_bank(bank):
                log(row, df.at[row, '사원번호'], '급여은행', bank, '숫자/특수문자 포함')
                df.at[row, '급여은행'] = '미입력'

    if '계좌번호' in df.columns:
        for row, raw in df['계좌번호'].items():
            account = str(raw).strip() if raw else ''
            if account in ('nan', 'NaN'):
                account = ''
            if not account:
                log(row, df.at[row, '사원번호'], '계좌번호', account, '결측')
                df.at[row, '계좌번호'] = '미입력'

    if '4대보험가입여부' in df.columns:
        for row, raw in df['4대보험가입여부'].items():
            insurance = str(raw).strip() if raw else ''
            if insurance in ('nan', 'NaN'):
                insurance = ''
            if not insurance or insurance not in INSURANCE_VALUES:
                log(row, df.at[row, '사원번호'], '4대보험가입여부', insurance, '결측/이상치')
                df.at[row, '4대보험가입여부'] = '미입력'

print('validate_salary 정의 완료')

validate_salary 정의 완료


### 2-7. 성과 정보 (성과점수 · 인사고과)

In [23]:
def validate_perf(df):
    if '성과점수' in df.columns:
        for row, raw in df['성과점수'].items():
            score = to_num(raw, '성과점수', row, df.at[row, '사원번호'])
            if score is None:
                df.at[row, '성과점수'] = '미입력'
            elif not (MIN_SCORE <= score <= MAX_SCORE):
                log(row, df.at[row, '사원번호'], '성과점수', raw, f'범위 초과 ({MIN_SCORE}~{MAX_SCORE})')
                df.at[row, '성과점수'] = '미입력'

    for year in PERF_YEARS:
        col = f'인사고과_{year}'
        if col not in df.columns:
            continue

        for row in df.index:
            grade_val = str(df.at[row, col]).strip() if df.at[row, col] else ''
            if grade_val in ('nan', 'NaN'):
                grade_val = ''
            emp_id = df.at[row, '사원번호']

            hire_date = valid_hire.get(row)
            hire_year = hire_date.year if hire_date else None

            retire_date_str = ''
            if '퇴직일자' in df.columns and df.at[row, '퇴직일자']:
                retire_date_str = str(df.at[row, '퇴직일자']).strip()
                if retire_date_str in ('nan', 'NaN'):
                    retire_date_str = ''
            retire_date = parse_date(retire_date_str)
            retire_year = retire_date.year if retire_date else None

            if hire_year and year < hire_year:
                df.at[row, col] = ''
                continue
            if retire_year and year > retire_year:
                df.at[row, col] = ''
                continue
            if hire_year and year == hire_year:
                if grade_val and grade_val not in PERF_GRADES:
                    log(row, emp_id, col, grade_val, '고정값 외')
                    df.at[row, col] = '미입력'
                continue
            if not grade_val:
                log(row, emp_id, col, grade_val, '재직 기간 내 결측')
                df.at[row, col] = '미입력'
            elif grade_val not in PERF_GRADES:
                log(row, emp_id, col, grade_val, '고정값 외')
                df.at[row, col] = '미입력'

print('validate_perf 정의 완료')

validate_perf 정의 완료


### 2-8. 자격 · 이력 (자격증 · TOEIC · 포상이력 · 징계이력)

In [24]:
def validate_qual(df):
    # TOEIC점수
    if 'TOEIC점수' in df.columns:
        for row, raw in df['TOEIC점수'].items():
            toeic_str = str(raw).strip() if raw else ''
            # 빈값이거나 nan이면 건너뜀 (TOEIC는 선택 항목)
            if not toeic_str or toeic_str in ('nan', 'NaN'):
                continue
            toeic = to_num(raw, 'TOEIC점수', row, df.at[row, '사원번호'])
            if toeic is None:
                df.at[row, 'TOEIC점수'] = '미입력'
            elif not (MIN_TOEIC <= toeic <= MAX_TOEIC):
                log(row, df.at[row, '사원번호'], 'TOEIC점수', raw, f'범위 초과 ({MIN_TOEIC}~{MAX_TOEIC})')
                df.at[row, 'TOEIC점수'] = '미입력'

    # 자격증수당여부
    if '자격증수당여부' in df.columns:
        for row, raw in df['자격증수당여부'].items():
            subsidy = str(raw).strip() if raw else ''
            if subsidy in ('nan', 'NaN'):
                subsidy = ''
            if not subsidy or subsidy not in SUBSIDY_VALUES:
                log(row, df.at[row, '사원번호'], '자격증수당여부', subsidy, '결측/이상치')
                df.at[row, '자격증수당여부'] = '미입력'

    # 자격증 / 포상이력 / 징계이력: 쉼표 구분 항목의 앞뒤 공백 정리
    for col in ['자격증', '포상이력', '징계이력']:
        if col not in df.columns:
            continue
        for row, raw in df[col].items():
            items = parse_array(raw)
            df.at[row, col] = ','.join(items)

    # 징계사유: 징계이력과 일관성 검사
    if '징계이력' in df.columns and '징계사유' in df.columns:
        for row in df.index:
            discipline_history = str(df.at[row, '징계이력']).strip()
            if discipline_history in ('nan', 'NaN'):
                discipline_history = ''
            discipline_reason = str(df.at[row, '징계사유']).strip() if df.at[row, '징계사유'] else ''
            if discipline_reason in ('nan', 'NaN'):
                discipline_reason = ''
            emp_id = df.at[row, '사원번호']

            # 징계이력 있는데 사유 없음
            if discipline_history and not discipline_reason:
                log(row, emp_id, '징계사유', discipline_reason, '징계이력 있는데 징계사유 없음')
                df.at[row, '징계사유'] = '미입력'
            # 징계이력 없는데 사유 존재
            elif not discipline_history and discipline_reason:
                log(row, emp_id, '징계사유', discipline_reason, '징계이력 없는데 징계사유 존재')
                df.at[row, '징계사유'] = ''

print('validate_qual 정의 완료')

validate_qual 정의 완료

# 3. 파일별 처리 및 저장

In [25]:
saved_files = []

# dfs 딕셔너리를 순회: 파일마다 독립적으로 정제 후 저장
for source_name, df in dfs.items():
    print(f'\n처리 중: {source_name}  ({len(df):,}행)')

    # 파일마다 전역 상태 초기화 (파일 간 오염 방지)
    _errors   = []
    drop_rows = set()
    valid_rrn  = {}
    valid_hire = {}

    # 검증 함수를 순서대로 실행 (앞 함수의 결과를 뒤 함수가 참조)
    validate_empid(df)
    validate_name(df)
    validate_rrn(df)
    validate_gender(df)
    validate_birth(df)
    validate_age(df)
    validate_military(df)
    validate_hire(df)
    validate_tenure(df)
    validate_edu(df)
    validate_career(df)
    validate_dept(df)
    validate_grade(df)
    validate_retire(df)
    validate_contact(df)
    validate_salary(df)
    validate_perf(df)
    validate_qual(df)

    # drop_rows에 담긴 행 번호를 제거하고 인덱스를 0부터 재정렬
    df_clean = df.drop(index=list(drop_rows)).reset_index(drop=True)

    # 정제된 데이터를 CSV로 저장
    out_path = OUTPUT_DIR / f'{source_name}_정제.csv'
    df_clean.to_csv(out_path, index=False, encoding='utf-8-sig')
    saved_files.append((out_path, len(df_clean)))
    print(f'  저장: {out_path.name}  ({len(df_clean):,}행 / 제거 {len(drop_rows)}행)')

    # 에러 로그가 있으면 파일명 앞에 원본 파일명을 붙여서 저장
    if _errors:
        error_path = OUTPUT_DIR / f'{source_name}_error.log'
        pd.DataFrame(_errors).to_csv(error_path, index=False, encoding='utf-8-sig')
        print(f'  에러 로그: {error_path.name}  ({len(_errors):,}건)')


처리 중: 급여정보  (2,000행)


  저장: 급여정보_정제.csv  (2,000행 / 제거 0행)

처리 중: 기본인사정보  (2,000행)


  저장: 기본인사정보_정제.csv  (2,000행 / 제거 0행)
  에러 로그: 기본인사정보_error.log  (2,284건)

처리 중: 역량성과  (2,000행)


  저장: 역량성과_정제.csv  (2,000행 / 제거 0행)
  에러 로그: 역량성과_error.log  (3,044건)

처리 중: 통합인사정보  (2,000행)


  저장: 통합인사정보_정제.csv  (2,000행 / 제거 0행)
  에러 로그: 통합인사정보_error.log  (2,344건)


In [26]:
print('=' * 60)
print('통합인사정보 데이터 전처리 완료')
print('=' * 60)
print('저장된 파일:')
for path, rows in saved_files:
    print(f'  {path.name}  ({rows:,}행)')
print('=' * 60)

통합인사정보 데이터 전처리 완료
저장된 파일:
  급여정보_정제.csv  (2,000행)
  기본인사정보_정제.csv  (2,000행)
  역량성과_정제.csv  (2,000행)
  통합인사정보_정제.csv  (2,000행)
